<a href="https://colab.research.google.com/github/Qudus-O/Financial-News-Sentiment/blob/main/finance_sentiment_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA vs. Full Fine-tuning for Financial News Sentiment

Follow-up to the DistilBERT vs. TF-IDF notebook (Same data, same
splits (stratified split, seed 42), same test set, same base model
(`distilbert-base-uncased`).

**Question:** how close does LoRA get to a full fine-tune, and what does it save?

**Compared on:** test macro-F1, number of trainable parameters, training time,
and size of the saved model.

##  Setup
Installs the libraries and prints their versions so the run can be reproduced.
`peft` is the library that provides LoRA. The seed is set here, and the training
seeds are set later in the training section.

In [1]:
!pip uninstall -y -q torchao
!pip install -q transformers datasets accelerate scikit-learn matplotlib peft

In [2]:

import re
import numpy as np
import torch, sklearn, transformers, datasets
from transformers import set_seed

set_seed(42)
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| sklearn", sklearn.__version__)
print("GPU available:", torch.cuda.is_available())

torch 2.11.0+cu130 | transformers 5.18.0 | datasets 4.8.5 | sklearn 1.6.1
GPU available: True


##  Data
Same dataset as the first notebook:
[`zeroshot/twitter-financial-news-sentiment`](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment)
(MIT license). Short financial news tweets with a 3-class sentiment label.

| Label | Sentiment |
|---|---|
| 0 | Bearish |
| 1 | Bullish |
| 2 | Neutral |

Neutral is about 65% of the training data. The file is not shuffled (the first rows
contain no neutral tweets), so the examples printed below are sampled randomly.
The tweets are loaded from the Hugging Face Hub at run time and are not stored in
this repository.

In [3]:
from collections import Counter
from datasets import load_dataset

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
LABELS = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

print(ds)
print({LABELS[k]: v for k, v in sorted(Counter(ds["train"]["label"]).items())})

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})
{'Bearish': 1442, 'Bullish': 1923, 'Neutral': 6178}


### Data quality check
After removing links and lowercasing, the cell below counts how many test texts also
appear in the training data. This is the same check as in the first notebook, and it
should give the same numbers: 38 of 2,388 test texts (1.6%) overlap, and 106
training rows duplicate other training rows. If your numbers differ, something
changed in the data or the code.

In [4]:
norm = lambda t: re.sub(r"https?://\S+", "", t).strip().lower()
train_texts = {norm(t) for t in ds["train"]["text"]}
dup = [t for t in ds["validation"]["text"] if norm(t) in train_texts]

print(len(dup), "of", len(ds["validation"]), "validation texts also appear in train")
print(dup[:3])
print(len(ds["train"]), "train rows,", len(train_texts), "unique after normalizing")

38 of 2388 validation texts also appear in train
['Fed Chairman Jerome Powell puts lawmakers on notice that fiscal policy may need to play a bigger role countering do… https://t.co/URFHNRkmfh', 'Exxon is cutting billions of dollars in investments to keep its financial footing during the worst oil-price rout i… https://t.co/MI5vCEkhlm', '3M is exploring a sale of its drug delivery systems unit, which could fetch about $1 billion https://t.co/ISlbwO3FCd']
9543 train rows, 9437 unique after normalizing


### Train / dev / test split
This must match the first notebook exactly, because the comparison with the full
fine-tune is only fair on the same data.

- **Test set:** the published validation set (2,388 rows), not used for any decision.
- **Dev set:** a stratified 10% of the training data (955 rows), used to monitor
  training only.
- **Fit set:** the remaining 8,588 rows, used for training.

Stratifying keeps the class proportions the same in each part. Split seed: 42.

In [5]:
raw_train = ds["train"].class_encode_column("label")
parts = raw_train.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
fit_raw, dev_raw = parts["train"], parts["test"]
test_raw = ds["validation"]
print(len(fit_raw), len(dev_raw), len(test_raw))

Stringifying the column:   0%|          | 0/9543 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/9543 [00:00<?, ? examples/s]

8588 955 2388


**What this shows:** the split sizes are 8,588 / 955 / 2,388, identical to the
first notebook.

### Duplicates and label conflicts
Checks whether any duplicated texts carry conflicting labels, and how many dev rows
also appear in the fit set. Same check as before.

In [6]:
train_labels = {}
for t, l in zip(ds["train"]["text"], ds["train"]["label"]):
    train_labels.setdefault(norm(t), set()).add(l)

test_dup_mask = np.array([norm(t) in train_labels for t in test_raw["text"]])
test_conflicts = sum(
    1 for t, l, d in zip(test_raw["text"], test_raw["label"], test_dup_mask)
    if d and l not in train_labels[norm(t)]
)
fit_norm = {norm(t) for t in fit_raw["text"]}

print("overlapping test rows:", int(test_dup_mask.sum()))
print("of those, label differs from every train label for that text:", test_conflicts)
print("train texts with more than one label:", sum(len(v) > 1 for v in train_labels.values()))
print("dev rows also in fit:", sum(norm(t) in fit_norm for t in dev_raw["text"]), "of", len(dev_raw))

overlapping test rows: 38
of those, label differs from every train label for that text: 1
train texts with more than one label: 2
dev rows also in fit: 16 of 955


**What this shows:** 38 test rows overlap with training, and only 1 of them has a
label that differs from the training label for the same text. Only 2 training texts
have more than one label. 16 of the 955 dev rows (1.7%) also appear in the fit set,
which makes dev scores slightly optimistic. Dev is used only to monitor training.

##  Tokenization
`distilbert-base-uncased` tokenizer, the same as before. Length statistics are
computed on the fit set only. `max_length=64` is chosen so that only a small share of
tweets is truncated. Padding is done per batch by the data collator, not here, to
avoid wasting compute on pad tokens.

In [7]:
from transformers import AutoTokenizer

MODEL = "distilbert-base-uncased"
MAX_LEN = 64
tokenizer = AutoTokenizer.from_pretrained(MODEL)

lengths = np.array([len(tokenizer(t)["input_ids"]) for t in fit_raw["text"]])
print("token length percentiles (50/90/99/max):", np.percentile(lengths, [50, 90, 99, 100]))
print("share truncated at", MAX_LEN, ":", round(float((lengths > MAX_LEN).mean()), 4))

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

fit_ds = fit_raw.map(tokenize, batched=True)
dev_ds = dev_raw.map(tokenize, batched=True)
test_ds = test_raw.map(tokenize, batched=True)
print(fit_ds)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

token length percentiles (50/90/99/max): [25. 47. 57. 82.]
share truncated at 64 : 0.0022


Map:   0%|          | 0/8588 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8588
})


**What this shows:** median tweet length is 25 tokens, the 99th percentile is 57, and
the longest is 82. With `max_length=64`, only 0.22% of fit tweets (roughly 19 of
8,588) are truncated.

## LoRA setup
LoRA freezes the pretrained weights and trains small extra matrices added to the
attention layers. The new classification head is trained in full.

In [8]:


from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)

cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
)

lora_model = get_peft_model(base, cfg)
lora_model.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


740,355 of 67,696,134 parameters (1.09%) are trainable. Only
147,456 of those are LoRA matrices (r=8, on the query and value projections in 6
layers). The other 592,899 belong to the classification head, which is trained in
full, so the head accounts for about 80% of trainable parameters in this small model.

## LoRA training (one seed first)
Same data, epochs (3), batch size (32) and max length (64) as the full fine-tune. The
one deliberate change is the learning rate: 2e-4 instead of 2e-5. LoRA's new matrices
start from scratch and usually need larger steps than full fine-tuning, and
1e-4 to 5e-4 is a common starting range. This is a single-seed check before running
three seeds.

In [9]:
import time
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
from sklearn.metrics import accuracy_score, f1_score

collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

set_seed(0)
base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
model = get_peft_model(base, cfg)

args = TrainingArguments(
    output_dir="lora-seed0", num_train_epochs=3,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=2e-4, eval_strategy="epoch", save_strategy="no",
    fp16=True, report_to="none", seed=0, label_names=["labels"])

trainer = Trainer(model=model, args=args, train_dataset=fit_ds, eval_dataset=dev_ds,
                  data_collator=collator, processing_class=tokenizer,
                  compute_metrics=compute_metrics)

t0 = time.time()
trainer.train()
train_secs = time.time() - t0

out = trainer.predict(test_ds)
print("train seconds:", round(train_secs, 1))
print("test acc", round(out.metrics["test_accuracy"], 4),
      "macro-F1", round(out.metrics["test_macro_f1"], 4))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.489923,0.811518,0.745730
2,0.552888,0.445901,0.832461,0.782991
3,0.552888,0.431226,0.835602,0.780720


train seconds: 64.9
test acc 0.8363 macro-F1 0.7784


In [10]:
from sklearn.metrics import classification_report

print(classification_report(out.label_ids, np.argmax(out.predictions, axis=-1),
                            target_names=["Bearish", "Bullish", "Neutral"], digits=3))

              precision    recall  f1-score   support

     Bearish      0.693     0.720     0.706       347
     Bullish      0.790     0.688     0.736       475
     Neutral      0.880     0.907     0.893      1566

    accuracy                          0.836      2388
   macro avg      0.788     0.772     0.778      2388
weighted avg      0.835     0.836     0.835      2388



## LoRA tuning (selected on dev, not test)
The first LoRA run (r=8, lr=2e-4, 3 epochs) was clearly below the full fine-tune. To
see why, I vary one setting at a time and compare dev macro-F1. The test set is not
used for these choices. All runs use seed 0 and `lora_alpha = 2 * r`.

In [11]:
def run_lora(name, r=8, lr=2e-4, epochs=3, targets=("q_lin", "v_lin"), seed=0):
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r, lora_dropout=0.1,
        target_modules=list(targets),
        modules_to_save=["pre_classifier", "classifier"])
    model = get_peft_model(base, cfg)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    args = TrainingArguments(
        output_dir=f"lora-{name}", num_train_epochs=epochs,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_strategy="no", fp16=True, report_to="none",
        seed=seed, label_names=["labels"], disable_tqdm=True)
    trainer = Trainer(model=model, args=args, train_dataset=fit_ds, eval_dataset=dev_ds,
                      data_collator=collator, processing_class=tokenizer,
                      compute_metrics=compute_metrics)
    t0 = time.time()
    trainer.train()
    secs = time.time() - t0
    curve = [round(h["eval_macro_f1"], 4) for h in trainer.state.log_history
             if "eval_macro_f1" in h]
    return {"name": name, "r": r, "lr": lr, "epochs": epochs,
            "targets": "+".join(t[0] for t in targets), "trainable": trainable,
            "secs": round(secs), "dev_last": curve[-1], "dev_curve": curve}

In [12]:
results = []
results.append(run_lora("base"))
results.append(run_lora("ep10", epochs=10))
for r_ in results:
    print(r_)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.4899', 'eval_accuracy': '0.8115', 'eval_macro_f1': '0.7457', 'eval_runtime': '0.8127', 'eval_samples_per_second': '1175', 'eval_steps_per_second': '18.46', 'epoch': '1'}
{'eval_loss': '0.4459', 'eval_accuracy': '0.8325', 'eval_macro_f1': '0.783', 'eval_runtime': '0.4341', 'eval_samples_per_second': '2200', 'eval_steps_per_second': '34.55', 'epoch': '2'}
{'eval_loss': '0.4312', 'eval_accuracy': '0.8356', 'eval_macro_f1': '0.7807', 'eval_runtime': '0.4406', 'eval_samples_per_second': '2168', 'eval_steps_per_second': '34.05', 'epoch': '3'}
{'train_runtime': '43.34', 'train_samples_per_second': '594.4', 'train_steps_per_second': '18.62', 'train_loss': '0.5094', 'epoch': '3'}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.4838', 'eval_accuracy': '0.8157', 'eval_macro_f1': '0.7526', 'eval_runtime': '0.4536', 'eval_samples_per_second': '2105', 'eval_steps_per_second': '33.07', 'epoch': '1'}
{'eval_loss': '0.4415', 'eval_accuracy': '0.8304', 'eval_macro_f1': '0.7832', 'eval_runtime': '0.461', 'eval_samples_per_second': '2072', 'eval_steps_per_second': '32.54', 'epoch': '2'}
{'eval_loss': '0.4032', 'eval_accuracy': '0.845', 'eval_macro_f1': '0.7897', 'eval_runtime': '0.49', 'eval_samples_per_second': '1949', 'eval_steps_per_second': '30.61', 'epoch': '3'}
{'eval_loss': '0.4005', 'eval_accuracy': '0.8492', 'eval_macro_f1': '0.8036', 'eval_runtime': '0.4769', 'eval_samples_per_second': '2003', 'eval_steps_per_second': '31.46', 'epoch': '4'}
{'eval_loss': '0.3812', 'eval_accuracy': '0.8524', 'eval_macro_f1': '0.8059', 'eval_runtime': '0.5161', 'eval_samples_per_second': '1850', 'eval_steps_per_second': '29.06', 'epoch': '5'}
{'eval_loss': '0.3788', 'eval_accuracy': '0.8503', 'eval_macro_f1': '

**What this shows (dev macro-F1, seed 0):** With 3 epochs, LoRA reaches 0.781, about
0.03 below the full fine-tune's dev scores (0.806 to 0.816). With 10 epochs it rises to
about 0.80 by epoch 4 to 5 and stays flat afterwards (0.800 to 0.806), so training
beyond about 5 epochs adds nothing. The remaining gap of about 0.01 is within the
noise of a 955-row dev set, so I cannot yet say whether LoRA has caught up.

## What limits LoRA: capacity or learning rate?
Epochs are fixed at 5 (the 10-epoch curve was flat after epoch 4 to 5). Each run
changes one setting from the 5-epoch control. All runs use seed 0 and are compared on
dev macro-F1. Differences below about 0.01 are not meaningful with a 955-row dev set.

In [13]:
results2 = []
results2.append(run_lora("ep5", epochs=5))
results2.append(run_lora("ep5_r16", r=16, epochs=5))
results2.append(run_lora("ep5_lr5e-4", lr=5e-4, epochs=5))
results2.append(run_lora("ep5_allattn", epochs=5, targets=("q_lin", "k_lin", "v_lin", "out_lin")))
for r_ in results2:
    print(r_["name"], "trainable", r_["trainable"], "secs", r_["secs"],
          "dev_last", r_["dev_last"], r_["dev_curve"])

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.486', 'eval_accuracy': '0.8105', 'eval_macro_f1': '0.7438', 'eval_runtime': '0.4718', 'eval_samples_per_second': '2024', 'eval_steps_per_second': '31.79', 'epoch': '1'}
{'eval_loss': '0.4458', 'eval_accuracy': '0.8314', 'eval_macro_f1': '0.7849', 'eval_runtime': '0.4704', 'eval_samples_per_second': '2030', 'eval_steps_per_second': '31.89', 'epoch': '2'}
{'eval_loss': '0.413', 'eval_accuracy': '0.8471', 'eval_macro_f1': '0.7936', 'eval_runtime': '0.4825', 'eval_samples_per_second': '1979', 'eval_steps_per_second': '31.09', 'epoch': '3'}
{'eval_loss': '0.4036', 'eval_accuracy': '0.8408', 'eval_macro_f1': '0.7881', 'eval_runtime': '0.4744', 'eval_samples_per_second': '2013', 'eval_steps_per_second': '31.62', 'epoch': '4'}
{'eval_loss': '0.3989', 'eval_accuracy': '0.8461', 'eval_macro_f1': '0.7986', 'eval_runtime': '0.4874', 'eval_samples_per_second': '1959', 'eval_steps_per_second': '30.78', 'epoch': '5'}
{'train_runtime': '61.37', 'train_samples_per_second': '699.7', 'tr

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.458', 'eval_accuracy': '0.8199', 'eval_macro_f1': '0.756', 'eval_runtime': '0.4766', 'eval_samples_per_second': '2004', 'eval_steps_per_second': '31.47', 'epoch': '1'}
{'eval_loss': '0.4081', 'eval_accuracy': '0.8325', 'eval_macro_f1': '0.7843', 'eval_runtime': '0.479', 'eval_samples_per_second': '1994', 'eval_steps_per_second': '31.31', 'epoch': '2'}
{'eval_loss': '0.3943', 'eval_accuracy': '0.8461', 'eval_macro_f1': '0.7904', 'eval_runtime': '0.4739', 'eval_samples_per_second': '2015', 'eval_steps_per_second': '31.65', 'epoch': '3'}
{'eval_loss': '0.3972', 'eval_accuracy': '0.8534', 'eval_macro_f1': '0.8034', 'eval_runtime': '0.4795', 'eval_samples_per_second': '1992', 'eval_steps_per_second': '31.28', 'epoch': '4'}
{'eval_loss': '0.4', 'eval_accuracy': '0.8555', 'eval_macro_f1': '0.8093', 'eval_runtime': '0.48', 'eval_samples_per_second': '1990', 'eval_steps_per_second': '31.25', 'epoch': '5'}
{'train_runtime': '51.44', 'train_samples_per_second': '834.7', 'train_st

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.4739', 'eval_accuracy': '0.8126', 'eval_macro_f1': '0.7381', 'eval_runtime': '0.5251', 'eval_samples_per_second': '1819', 'eval_steps_per_second': '28.57', 'epoch': '1'}
{'eval_loss': '0.4365', 'eval_accuracy': '0.8356', 'eval_macro_f1': '0.7901', 'eval_runtime': '0.525', 'eval_samples_per_second': '1819', 'eval_steps_per_second': '28.57', 'epoch': '2'}
{'eval_loss': '0.3988', 'eval_accuracy': '0.8429', 'eval_macro_f1': '0.7903', 'eval_runtime': '0.5308', 'eval_samples_per_second': '1799', 'eval_steps_per_second': '28.26', 'epoch': '3'}
{'eval_loss': '0.3945', 'eval_accuracy': '0.8503', 'eval_macro_f1': '0.7978', 'eval_runtime': '0.5328', 'eval_samples_per_second': '1792', 'eval_steps_per_second': '28.15', 'epoch': '4'}
{'eval_loss': '0.3864', 'eval_accuracy': '0.8513', 'eval_macro_f1': '0.8059', 'eval_runtime': '0.5358', 'eval_samples_per_second': '1782', 'eval_steps_per_second': '27.99', 'epoch': '5'}
{'train_runtime': '61.07', 'train_samples_per_second': '703.1', 't

**What this shows (dev macro-F1, seed 0, 5 epochs):** the control scores 0.799. Doubling
r to 16 gives 0.806, a learning rate of 5e-4 gives 0.809, and adapting all four
attention projections gives 0.806. All variants are within 0.011 of the control, which
is about the size of run-to-run noise on a 955-row dev set (the 10-epoch run scored
0.806 at epoch 5 with the same seed). r=16 and all-attention have the same number of
trainable parameters (887,811) and score the same. I pick lr=5e-4 because it is the
highest on dev, but the margin is within noise.

## LoRA final runs
Chosen on dev macro-F1: r=8, learning rate 5e-4, 5 epochs, query and value
projections. The choice was among four settings and the margin was within noise.
Three seeds (0, 1, 2); test set scored once per seed.

In [14]:
import pandas as pd

def final_lora(seed, r=8, lr=5e-4, epochs=5):
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r, lora_dropout=0.1,
                     target_modules=["q_lin", "v_lin"],
                     modules_to_save=["pre_classifier", "classifier"])
    model = get_peft_model(base, cfg)
    args = TrainingArguments(
        output_dir=f"lora-final-{seed}", num_train_epochs=epochs,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_strategy="no", fp16=True, report_to="none",
        seed=seed, label_names=["labels"], disable_tqdm=True)
    trainer = Trainer(model=model, args=args, train_dataset=fit_ds, eval_dataset=dev_ds,
                      data_collator=collator, processing_class=tokenizer,
                      compute_metrics=compute_metrics)
    t0 = time.time()
    trainer.train()
    secs = time.time() - t0
    dev = trainer.evaluate()
    out = trainer.predict(test_ds)
    return {"seed": seed, "secs": round(secs), "dev_macro_f1": dev["eval_macro_f1"],
            "test_acc": out.metrics["test_accuracy"],
            "test_macro_f1": out.metrics["test_macro_f1"],
            "preds": np.argmax(out.predictions, axis=-1)}

lora_runs = [final_lora(s) for s in [0, 1, 2]]
lora_df = pd.DataFrame([{k: v for k, v in r_.items() if k != "preds"} for r_ in lora_runs])
print(lora_df.round(4).to_string(index=False))
print("test acc      mean", round(lora_df.test_acc.mean(), 4), "std", round(lora_df.test_acc.std(ddof=0), 4))
print("test macro-F1 mean", round(lora_df.test_macro_f1.mean(), 4), "std", round(lora_df.test_macro_f1.std(ddof=0), 4))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.458', 'eval_accuracy': '0.8199', 'eval_macro_f1': '0.756', 'eval_runtime': '0.4857', 'eval_samples_per_second': '1966', 'eval_steps_per_second': '30.89', 'epoch': '1'}
{'eval_loss': '0.4081', 'eval_accuracy': '0.8325', 'eval_macro_f1': '0.7843', 'eval_runtime': '0.4704', 'eval_samples_per_second': '2030', 'eval_steps_per_second': '31.89', 'epoch': '2'}
{'eval_loss': '0.3943', 'eval_accuracy': '0.8461', 'eval_macro_f1': '0.7904', 'eval_runtime': '0.4857', 'eval_samples_per_second': '1966', 'eval_steps_per_second': '30.88', 'epoch': '3'}
{'eval_loss': '0.3972', 'eval_accuracy': '0.8534', 'eval_macro_f1': '0.8034', 'eval_runtime': '0.472', 'eval_samples_per_second': '2023', 'eval_steps_per_second': '31.78', 'epoch': '4'}
{'eval_loss': '0.4', 'eval_accuracy': '0.8555', 'eval_macro_f1': '0.8093', 'eval_runtime': '0.4692', 'eval_samples_per_second': '2035', 'eval_steps_per_second': '31.97', 'epoch': '5'}
{'train_runtime': '51.59', 'train_samples_per_second': '832.3', 'train_

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.449', 'eval_accuracy': '0.8209', 'eval_macro_f1': '0.757', 'eval_runtime': '0.4728', 'eval_samples_per_second': '2020', 'eval_steps_per_second': '31.73', 'epoch': '1'}
{'eval_loss': '0.3995', 'eval_accuracy': '0.8408', 'eval_macro_f1': '0.781', 'eval_runtime': '0.4798', 'eval_samples_per_second': '1990', 'eval_steps_per_second': '31.26', 'epoch': '2'}
{'eval_loss': '0.3917', 'eval_accuracy': '0.8377', 'eval_macro_f1': '0.7887', 'eval_runtime': '0.4749', 'eval_samples_per_second': '2011', 'eval_steps_per_second': '31.59', 'epoch': '3'}
{'eval_loss': '0.3879', 'eval_accuracy': '0.8524', 'eval_macro_f1': '0.8023', 'eval_runtime': '0.4834', 'eval_samples_per_second': '1976', 'eval_steps_per_second': '31.03', 'epoch': '4'}
{'eval_loss': '0.3916', 'eval_accuracy': '0.8492', 'eval_macro_f1': '0.8025', 'eval_runtime': '0.4734', 'eval_samples_per_second': '2017', 'eval_steps_per_second': '31.69', 'epoch': '5'}
{'train_runtime': '51.68', 'train_samples_per_second': '830.8', 'tra

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': '0.4432', 'eval_accuracy': '0.8293', 'eval_macro_f1': '0.7688', 'eval_runtime': '0.4821', 'eval_samples_per_second': '1981', 'eval_steps_per_second': '31.11', 'epoch': '1'}
{'eval_loss': '0.4148', 'eval_accuracy': '0.8356', 'eval_macro_f1': '0.7911', 'eval_runtime': '0.4802', 'eval_samples_per_second': '1989', 'eval_steps_per_second': '31.23', 'epoch': '2'}
{'eval_loss': '0.3843', 'eval_accuracy': '0.8586', 'eval_macro_f1': '0.8132', 'eval_runtime': '0.4722', 'eval_samples_per_second': '2022', 'eval_steps_per_second': '31.76', 'epoch': '3'}
{'eval_loss': '0.3955', 'eval_accuracy': '0.8503', 'eval_macro_f1': '0.8008', 'eval_runtime': '0.4748', 'eval_samples_per_second': '2011', 'eval_steps_per_second': '31.59', 'epoch': '4'}
{'eval_loss': '0.3966', 'eval_accuracy': '0.8513', 'eval_macro_f1': '0.8044', 'eval_runtime': '0.4752', 'eval_samples_per_second': '2010', 'eval_steps_per_second': '31.57', 'epoch': '5'}
{'train_runtime': '51.58', 'train_samples_per_second': '832.4', '

**What this shows (test set, 3 seeds):** LoRA (r=8, lr 5e-4, 5 epochs) reaches 0.862
accuracy and 0.819 macro-F1 (seed-to-seed std 0.004), against 0.874 and 0.835 for the
full fine-tune. The macro-F1 gap of 0.016 is about four times LoRA's own seed spread,
so it is unlikely to be noise, although three seeds is a small sample. On dev the two
methods were only 0.007 apart, which a 955-row dev set cannot separate from noise, so
the larger test set was needed to see the gap. LoRA's settings were tuned on dev among
four options, while the full fine-tune used untuned defaults and still scored higher.

## Cost comparison: LoRA vs. full fine-tune
Both methods are trained for 3 epochs on the fit set, seed 0, each measured twice,
alternating. Peak GPU memory is measured relative to the memory already in use before
each run. Saved size is the size of the files written by `save_pretrained`. Learning
rates are each method's own (2e-5 full, 5e-4 LoRA).

In [15]:
import os, shutil, gc

def measure(kind, epochs=3, seed=0):
    gc.collect(); torch.cuda.empty_cache()
    base_mem = torch.cuda.memory_allocated()
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    if kind == "lora":
        cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                         target_modules=["q_lin", "v_lin"],
                         modules_to_save=["pre_classifier", "classifier"])
        model, lr = get_peft_model(base, cfg), 5e-4
    else:
        model, lr = base, 2e-5
    args = TrainingArguments(
        output_dir=f"measure-{kind}", num_train_epochs=epochs,
        per_device_train_batch_size=32, learning_rate=lr, eval_strategy="no",
        save_strategy="no", logging_strategy="no", fp16=True, report_to="none",
        seed=seed, label_names=["labels"], disable_tqdm=True)
    trainer = Trainer(model=model, args=args, train_dataset=fit_ds,
                      data_collator=collator, processing_class=tokenizer)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    secs = time.time() - t0
    peak = (torch.cuda.max_memory_allocated() - base_mem) / 2**20
    path = f"saved-{kind}"
    shutil.rmtree(path, ignore_errors=True)
    model.save_pretrained(path)
    size = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)) / 2**20
    del trainer, model, base
    return {"method": kind, "secs": round(secs, 1), "peak_gpu_MB": round(peak),
            "saved_MB": round(size, 1)}

cost = pd.DataFrame([measure(k) for k in ["full", "lora", "full", "lora"]])
print(cost.to_string(index=False))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'train_runtime': '43.1', 'train_samples_per_second': '597.8', 'train_steps_per_second': '18.72', 'train_loss': '0.392', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

{'train_runtime': '43.27', 'train_samples_per_second': '595.4', 'train_steps_per_second': '18.65', 'train_loss': '0.392', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'train_runtime': '29.7', 'train_samples_per_second': '867.4', 'train_steps_per_second': '27.17', 'train_loss': '0.4532', 'epoch': '3'}
method  secs  peak_gpu_MB  saved_MB
  full  43.4         1288     255.4
  lora  29.9          671       2.8
  full  43.5         1288     255.4
  lora  30.0          671       2.8


**What this shows (3 epochs, seed 0, each method measured twice, same GPU):**

| | Full fine-tune | LoRA (r=8) |
|---|---|---|
| Training time | 43.4 s | 29.9 s |
| Peak GPU memory | 1,288 MB | 671 MB |
| Saved size | 255.4 MB | 2.8 MB |

At equal epochs LoRA is about 31% faster and uses about 48% less GPU memory. But LoRA
needed 5 epochs to reach its final score (about 50 s), while the full fine-tune used 3
(43 s), so the time advantage disappears for equal quality, and LoRA still scores lower
(0.819 vs 0.835 macro-F1). About 80% of the 2.8 MB LoRA file is the classification head.
On a 67M-parameter model these savings are small in absolute terms. They matter much
more for models with billions of parameters.

In [16]:
from sklearn.metrics import precision_recall_fscore_support

y = np.array(test_raw["label"])
per = np.array([
    np.stack(precision_recall_fscore_support(y, r_["preds"], labels=[0, 1, 2],
                                             zero_division=0)[:3])
    for r_ in lora_runs])
tbl = pd.DataFrame({
    "class": ["Bearish", "Bullish", "Neutral"],
    "precision": per[:, 0].mean(0), "recall": per[:, 1].mean(0),
    "f1": per[:, 2].mean(0), "recall_std": per[:, 1].std(0)})
print(tbl.round(3).to_string(index=False))

  class  precision  recall    f1  recall_std
Bearish      0.736   0.784 0.759       0.015
Bullish      0.801   0.780 0.791       0.011
Neutral      0.910   0.904 0.907       0.006


**What this means (test set, mean of 3 seeds each):**

| Class | Metric | Full fine-tune* | LoRA | Difference |
|---|---|---|---|---|
| Bearish | precision | 0.767 | 0.736 | -0.031 |
| | recall | 0.804 | 0.784 | -0.020 |
| | F1 | 0.785 | 0.759 | -0.026 |
| Bullish | precision | 0.816 | 0.801 | -0.015 |
| | recall | 0.796 | 0.780 | -0.016 |
| | F1 | 0.806 | 0.791 | -0.015 |
| Neutral | precision | 0.916 | 0.910 | -0.006 |
| | recall | 0.913 | 0.904 | -0.009 |
| | F1 | 0.914 | 0.907 | -0.007 |

*Full fine-tune figures are from the first notebook (same data, splits and test set).

LoRA is slightly weaker on every class. The largest loss is on Bearish (F1 -0.026),
mostly through lower precision. The recall gaps (0.016 to 0.020) are only 1 to 1.5
times the seed-to-seed spread of LoRA's recall (0.011 to 0.015), so they are likely
small.

## Conclusions
- LoRA (r=8, learning rate 5e-4, 5 epochs) reaches 0.819 macro-F1 over 3 seeds, against
  0.835 for the full fine-tune and 0.690 for TF-IDF. It recovers most of the quality of
  full fine-tuning, but not all of it.
- The gap is spread across classes and is largest on Bearish (F1 -0.026).
- Tuning mattered. The first LoRA run (one seed, learning rate 2e-4, 3 epochs) scored
  0.778. More epochs and a higher learning rate recovered about 0.04 of the 0.057 gap.
- Cost, at equal epochs: LoRA used about 31% less time, about 48% less peak GPU memory,
  and a saved file 91 times smaller (2.8 MB vs 255 MB). To reach its final score it
  needed 5 epochs, so its total training time (about 50 s) was slightly longer than the
  full fine-tune's 3 epochs (43 s).
- On this 67M-parameter model the savings are small in absolute terms. They should
  matter more as models grow.

**Limitations:** one dataset; 3 seeds per method; LoRA settings were chosen on dev among
four options with a margin within noise, while the full fine-tune used default settings;
cost was measured on one GPU type, one model size, one seed and 3 epochs.

## Summary for non-technical readers
Tested whether an AI model can flag bad news in short financial posts. A simple
word-counting method caught about 41 of every 100 bad-news posts. A fine-tuned AI model
caught about 80, at the cost of more false alarms (about 9 of every 100 harmless posts
were wrongly flagged, up from about 4). A cheaper way to train the model, called LoRA,
caught about 78 and used less memory and storage, but on a model this small the
savings were minor. The results come from one dataset of English financial posts and
should not be used for trading decisions.